# C1.1 · The agentic offensive workflow, and containing it

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Both directions*

Builds on **[C1.0 · Start here — what red teaming and research with AI means](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**.

| | |
|---|---|
| Tools used | CAI, Metasploit, Firecracker, Kimi K2, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Drive a planner/executor pair against a local target and watch the scope guard refuse an out-of-scope host before the request leaves.

**Why a security engineer needs it.** Payload suggestions instead of attack chains — and an offensive loop with no hard scope enforcement, which is an incident with a project plan. The control it builds is: full target context before it swings, and scope enforced at the network layer rather than by a politeness clause in the prompt.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An offensive harness reads only hostile input, by definition: every byte comes from a system you are attacking. It is the most dangerous agent in the building, and the thing that makes running it professional is that scope stops living in the tester's attention.

> **At CyberTravels.** An offensive loop pointed at CyberTravels' staging estate is the most dangerous thing in the building — and the engagement scope has to be enforced below the model, because everything the harness reads comes from the system it is attacking.

## 2 · The framework

```
   recon --> hypothesis --> test --> escalate --> report
        (the loop has not changed; who runs each turn has)

   +--------------------------------------------------+
   |  harness scope check   host in engagement set?   |
   +--------------------------------------------------+
   |  sandbox egress        private/link-local? rate? |
   +--------------------------------------------------+
              two layers, neither of them the model

   everything the harness reads is hostile by design
```

Penetration testing has always been a loop: **recon → hypothesis → test →
escalate → report.** What has changed is who runs each turn.

**Manual (still the baseline).** A human runs `nmap`, reads the output, forms a
hypothesis, tries it. Slow, and the quality is entirely the tester's.

**Scripted.** The recon is automated — Nuclei templates, a Burp scan. The
hypothesis and the escalation are still human. This is where most teams are.

**Semi-autonomous.** An open-weight model reads the recon output and *proposes*
which findings are worth chasing and what to try next. The human approves each
action. The gain is triage speed on a large surface: 400 findings ranked in
minutes rather than a day.

**Autonomous.** The model proposes and the harness executes, within a
pre-approved scope and tool set, verifying its own results. This is real and it
works, and it is also where the engagement becomes a safety problem — because an
agent that has not understood the scope will happily test something outside it
at machine speed.

The professional obligations do not change with autonomy. They get harder,
because scope enforcement can no longer live in the tester's attention — it has
to live in the harness, and then underneath the harness in the network.

That second half is why containment belongs in this lesson rather than in a
later one. An offensive harness has a property no other agent has: **everything
it reads is hostile by design.** Banner strings, error bodies, file contents —
all of it comes from a system you are attacking, which may itself already be
attacker-controlled. Containment there protects three parties at once: the
client (scope and rate limits, so you do not break their production), everyone
else (egress control, so a compromised harness cannot pivot outward), and you
(findings and client data must not leave by a route the agent chooses).

## 3 · Demo — the four generations on the same recon output

Realistic scan output from an authorised engagement against hosts you own. The question at every generation is the same: what do I chase first?

## 4 · Where it breaks — generation 4, and the scope problem

The model's top-ranked item is correct. Its reasoning on F-06 is also correct — *out of scope, do not touch*. Now make it autonomous and remove the human from the loop. What stops it acting on a finding it has correctly identified as out of scope?

Nothing in the model. Its judgement about scope is a *proposal*, on the decision plane, exactly like everything else it produces.

## 5 · The control — and the layer underneath it

The scope check above lives in the harness, which is one process away from the loop it constrains. On an engagement a single control is a single point of failure, and the failure is a professional incident. The same rule therefore gets restated where the agent cannot reach it: the sandbox's own request path.

## 6 · The procedure, as a skill

Model triage beats severity sorting on CyberTravels' findings and correctly calls the partner CDN out of scope — and can be argued into calling it critical. The skill runs both, then re-runs with scope enforced outside the model, where the persuaded model still proposes the call and nothing acts on it.

### The skill — [`skills/redteam/offensive-agent-containment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/offensive-agent-containment/SKILL.md)

```yaml
name: offensive-agent-containment
description: >-
  Run an offensive agent's triage inside an enforced engagement scope, and check
  what the enforcement does when the model is adversarially convinced that an
  out-of-scope host is critical. Use when giving an agent offensive capability,
  or when scope is currently a sentence in a statement of work.
allowed-tools: Read, Grep, Glob
```

# Scope enforced outside the model, or not enforced

A model triaging pentest findings beats severity sorting: it reads the finding
and reasons about exploitability, including that the partner CDN is out of
scope. That is a good reason to use one and a bad reason to trust it, because
the same reasoning can be argued with. Containment is the part that cannot.

## When to use this

Any agent with offensive capability — scanning, exploitation, recon — and any
workflow where scope is enforced by asking the model to respect it.

## Procedure

**1 — Establish the baseline you are improving on.** Sort by severity, take the
top *n*, and count how many exploitable findings you caught. This is the number
model triage has to beat, and it is usually beaten.

**2 — Run the model's triage and record its reasoning.** Including the scope
call. Note that it is correct — the argument here is not that the model is bad.

**3 — Adversarially convince it.** Craft the finding so the out-of-scope host
looks critical. A capable model will be persuaded, because being persuadable by
evidence is what makes it useful.

**4 — Re-run with scope enforced outside the model.** An allow-list of hosts,
plus a check on private ranges and the metadata address, evaluated on the action
rather than on the plan. The persuaded model still proposes it; nothing acts on
it.

**5 — Report both runs.** Unenforced and enforced, on the same findings. The
comparison is the deliverable: the model's judgement improved triage and did not
provide containment, and those are separate purchases.

## Example

**Input** — the fixture committed at the top of [`scripts/offensive_agent_containment.py`](scripts/offensive_agent_containment.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
=== generation 1: manual — a human reads all six and decides ===
   6 findings, no ordering, ~20 min of reading

=== generation 2: scripted — sort by severity ===
   F-02 high    /v1/users returns data without auth
   F-01 medium  TLS 1.0 enabled
   F-04 medium  password auth permitted
   → severity is a label, not a prediction. F-04 and F-05 are both
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "baseline": {"method": "severity", "top_n": 0, "exploitable_found": 0},
  "model_triage": {"top_n": 0, "exploitable_found": 0, "scope_calls": [{"host": "str", "in_scope": false}]},
  "adversarial": {"payload": "str", "model_convinced": true},
  "enforced": {"scope": ["str"], "blocked": ["str"], "reason": ["allow-list", "private range", "metadata"]},
  "conclusion": {"triage_improved": true, "containment_from_model": false}
}
```

## Failure modes

- **Enforcing scope in the prompt.** It is a request, and the adversarial case
  is precisely one that argues with requests.
- **Checking the plan rather than the action.** The plan is text; the action is
  where the allow-list applies.
- **Concluding the model is untrustworthy.** It improved triage. It is not a
  control, which is a different sentence.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/offensive-agent-containment/scripts/offensive_agent_containment.py
SCRIPT = "skills/redteam/offensive-agent-containment/scripts/offensive_agent_containment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Severity sorting puts 2 of 3 exploitable findings in the top 3; model triage puts 3 of 3, and correctly reasons that the partner CDN is out of scope. With the model adversarially convinced that the out-of-scope host is critical, the unenforced harness acts on it and the enforced harness refuses. Underneath the harness the sandbox refuses three requests for three different reasons — rate limit, engagement boundary, and cloud metadata — without consulting the model at all.

## Your turn

Write your engagement scope as a data structure your harness reads, not as a paragraph in a PDF. Then ask what your current tooling would do if a target redirected to a host you were not authorised to touch.

---

**Next → [C1.2 · Red-teaming an agent: designing the campaign](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*